# Customer churn analysis
Synthetic subscription data. Run cells in order after running `python -m src.pipeline` from the project root. A notebook kernel needs pandas, matplotlib and IPython/Jupyter; VS Code may offer to install a kernel.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import json, sqlite3
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
df = pd.read_csv(ROOT / 'data/customers_clean.csv')
historical = df[df.snapshot_date < '2025-11-01']
historical.head()

## Quality and cohorts
Raw source includes 20 duplicates. Missing values stay in the cleaned data and are imputed only inside training preprocessing.

In [ ]:
print((ROOT/'reports/data_quality.json').read_text())
historical.groupby('snapshot_date').agg(customers=('customer_id','size'), churn_rate=('churn_next_30d','mean'))

## Exploratory analysis
These associations are deliberately encoded in the simulator and do not establish causal effects.

In [ ]:
historical.groupby('contract').churn_next_30d.mean().plot.bar(title='Historical synthetic churn by contract')
plt.ylabel('Churn fraction'); plt.show()
historical.select_dtypes('number').describe()

## SQL analysis

In [ ]:
with sqlite3.connect(ROOT/'data/churn.db') as con:
    summary = pd.read_sql_query('SELECT contract, COUNT(*) AS customers, AVG(churn_next_30d) AS churn_rate FROM historical_customers GROUP BY contract',con)
summary

## Selection and holdout results
Select on July, test once on September; Jan/Mar/May train. Outcome maturity is preserved by gaps between cohorts.

In [ ]:
display(pd.read_csv(ROOT/'reports/model_comparison_validation.csv'))
json.loads((ROOT/'reports/metrics.json').read_text())

## Global importance and outreach
Importance is computed on validation data; recommended actions are separate rules, not per-customer model explanations.

In [ ]:
display(pd.read_csv(ROOT/'reports/feature_importance.csv'))
pd.read_csv(ROOT/'data/retention_outreach.csv').head(10)

## Conclusions
Read the generated findings below. Next step: validate on real historical data and measure retention interventions using a randomized experiment.

In [ ]:
print((ROOT/'reports/RESULTS.md').read_text())